# 16 · Running a model locally: Ollama and LM Studio
Level L2 · Part 2 · professional · ~30 min · builds on 07

## Goal
Run the small open model `granite4.1:3b` with Ollama, call it from Python and through an
OpenAI-compatible endpoint, ask it the same two Scout questions as Gemini, and print a side-by-side
verdict on answers, speed and tool choice that says where the small local model falls short.

## The idea
An **open-weight model** is a model whose weights you can download. A **local runner** such as Ollama
or LM Studio loads those weights into your laptop's memory and serves them over HTTP on `localhost`.
That is **local inference**: no key, no bill per token, and your notes never leave the machine.
To fit in memory the weights are **quantised**: each number is stored in about 4 bits instead of 16.
The price is quality and speed: a 3-billion-parameter model on a CPU is slower and makes more mistakes
than a hosted model, so you measure before you trust it.

Analogy: a hosted model is a restaurant kitchen; a local model is cooking at home with a smaller stove.

## Picture
```mermaid
flowchart LR
    P["Python (ollama or openai client)"] -->|"HTTP :11434"| O["Ollama server"]
    P -->|"HTTP :1234/v1"| L["LM Studio server"]
    O --> W["granite4.1:3b weights<br/>(4-bit, 2.1 GB)"]
    L --> W2["any GGUF model"]
    P -->|"HTTPS"| G["Gemini (hosted)"]
```

## Step by step
### 1. Start the recorder
Local calls are HTTP calls too, so `llm_replay` records them exactly like Gemini calls (lesson 07).
The model's replies and its own timing numbers were recorded from a real Ollama server; replay needs
neither Ollama nor a key.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("16_local-models")

LLM mode: replay (16_local-models.yaml)


### 2. Setup from lessons 07 and 08
Scout's notes, the system instruction that grounds answers in them (lesson 08), the Gemini client and
pinned model (lesson 07), and two test questions, each with a check that reads the answer: an easy
look-up (note 5) and one multiplication (24 h 39 min x 10 = 246.5 hours).

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
NOTES_TEXT = "\n".join(f"Note {n['id']}: {n['text']}" for n in NOTES)
SYSTEM_TEXT = ("You are Scout, a research assistant. Answer only from the user's notes below and this chat. "
               "If the notes do not cover it, say so in one sentence. Keep answers to one sentence.\n\n" + NOTES_TEXT)
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
TASKS = {
    "lookup": ("Which note is about a spacecraft, not a planet? Give its number.", lambda a: "5" in a),
    "math":   ("Using note 3, how many Earth hours do 10 Martian days take? Work it out and show the number.", lambda a: "246" in a),
}
print("hosted:", MODEL, "| tasks:", list(TASKS))

hosted: gemini-3.5-flash-lite | tasks: ['lookup', 'math']


### 3. Install Ollama and pull the model (once, on your machine)
Install Ollama from ollama.com (macOS/Windows installer, or the Linux script). It runs a server on
`http://localhost:11434`. Then download the model: `granite4.1:3b` is IBM's 3B open model, 2.1 GB, with
tool calling. This cell is tagged `needs-download` and `slow`, so it is skipped unless you enable it.

In [3]:
import subprocess
for cmd in (["ollama", "pull", "granite4.1:3b"], ["ollama", "list"]):
    out = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    print("$", " ".join(cmd), "->", out.stdout.strip().splitlines()[-1] if out.stdout.strip() else out.stderr.strip())

⏭  Skipped: this cell needs [download,slow]. Run it yourself, or re-run with TUT_ENABLE=download,slow.


### 4. What you downloaded
The `ollama` Python package talks to that server. `show` describes a model and `list` gives its size on
disk. Notice `quantization_level`: that is the 4-bit storage from The idea.

In [4]:
import ollama

LOCAL_MODEL = "granite4.1:3b"
local = ollama.Client(host="http://localhost:11434")
info = local.show(LOCAL_MODEL)
size_bytes = next(m.size for m in local.list().models if m.model == LOCAL_MODEL)
print("format      :", info.details.format, "| family:", info.details.family)
print("parameters  :", info.details.parameter_size)
print("quantisation:", info.details.quantization_level)
print("capabilities:", info.capabilities)
print(f"size on disk: {size_bytes / 1e9:.2f} GB")

format      : gguf | family: granite
parameters  : 3.4B
quantisation: Q4_K_M
capabilities: ['completion', 'tools']
size on disk: 2.10 GB


**GGUF** is the single-file weight format that Ollama and LM Studio both load (from the llama.cpp
project). `Q4_K_M` means about 4 bits per weight. Why that matters: memory.

### 5. Quantisation by hand
Quantising stores each weight as a small integer plus one shared **scale**. Here are 8 made-up weights
squeezed into 4-bit integers (-7 to 7) and expanded back.

In [5]:
weights = [0.82, -0.41, 0.05, -1.30, 0.67, 0.12, -0.88, 1.04]
scale = max(abs(w) for w in weights) / 7                 # one scale for the whole group
q4 = [round(w / scale) for w in weights]                 # what is stored: 4-bit integers
restored = [q * scale for q in q4]
print("4-bit ints:", q4)
print("restored  :", [round(r, 2) for r in restored])
print("max error :", round(max(abs(w - r) for w, r in zip(weights, restored)), 3))

4-bit ints: [4, -2, 0, -7, 4, 1, -5, 6]
restored  : [0.74, -0.37, 0.0, -1.3, 0.74, 0.19, -0.93, 1.11]
max error : 0.077


Each weight moved by at most half a step. Real formats like `Q4_K_M` use small groups, extra scales and
keep a few sensitive layers at higher precision, but the idea is this. Now the memory arithmetic for
3.4 billion weights at each precision, against the real file.

In [6]:
PARAMS = 3.4e9
for name, bits in [("float32", 32), ("float16", 16), ("8-bit", 8), ("4-bit", 4)]:
    print(f"{name:<8}{PARAMS * bits / 8 / 1e9:6.1f} GB")
print(f"real Q4_K_M file: {size_bytes / 1e9:.1f} GB = {size_bytes * 8 / PARAMS:.1f} bits per weight")

float32   13.6 GB
float16    6.8 GB
8-bit      3.4 GB
4-bit      1.7 GB
real Q4_K_M file: 2.1 GB = 4.9 bits per weight


Float16 would need 6.8 GB of RAM before Scout even starts; the 4-bit file needs about 2 GB. The real
file averages a little under 5 bits per weight because of the scales and the higher-precision layers.

### 6. First local call
`chat` takes the same `messages` list as lesson 08 (system, then user). `options` sets sampling:
`temperature` 0 (lesson 04) and `num_predict`, Ollama's name for the output-token cap.

In [7]:
def ask_local(question: str):
    messages = [{"role": "system", "content": SYSTEM_TEXT}, {"role": "user", "content": question}]
    return local.chat(model=LOCAL_MODEL, messages=messages, options={"temperature": 0, "num_predict": 200})

r = ask_local(TASKS["lookup"][0])
print("answer     :", r.message.content)
print("done_reason:", r.done_reason, "| tokens in/out:", r.prompt_eval_count, "/", r.eval_count)

answer     : 5
done_reason: stop | tokens in/out: 161 / 2


The reply object has the same parts as Gemini's: text, a stop reason and token counts, under other
names. Ollama also reports its own timings, in **nanoseconds**, and those are recorded too.

### 7. Where the time goes
Three phases: **load** (weights from disk into RAM, only when the model is cold), **prompt eval**
(reading the input) and **eval** (writing the output, token by token).

In [8]:
NS = 1e9

def timings(resp) -> dict:
    return {"load_s": resp.load_duration / NS, "prompt_s": resp.prompt_eval_duration / NS,
            "gen_s": resp.eval_duration / NS, "total_s": resp.total_duration / NS,
            "gen_tok_per_s": resp.eval_count / (resp.eval_duration / NS)}

print({k: round(v, 2) for k, v in timings(r).items()})

{'load_s': 4.87, 'prompt_s': 8.96, 'gen_s': 0.16, 'total_s': 14.0, 'gen_tok_per_s': 12.12}


This first call was cold: loading the weights and reading the prompt took nearly all the time, and
writing the short answer took a fraction of a second. Ollama keeps the model in memory for a few minutes
after a call (its keep-alive), so the next calls skip the load. These numbers come from a
4-core cloud VM with no GPU; a laptop with Apple silicon or a GPU is several times faster.

### 8. The same questions, local and hosted
Now both tasks on both models. Gemini's timing is measured with `time.perf_counter()` (lesson 11);
Ollama's comes from its own `total_duration`.

In [9]:
import time

GEMINI_CONFIG = types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, max_output_tokens=256,
                                            temperature=0, automatic_function_calling=NO_AFC)
rows = []
for task, (question, check) in TASKS.items():
    lr = ask_local(question)
    rows.append({"task": task, "model": LOCAL_MODEL, "answer": lr.message.content.strip(),
                 "seconds": round(lr.total_duration / NS, 2), "ok": check(lr.message.content)})
    start = time.perf_counter()
    gr = client.models.generate_content(model=MODEL, contents=question, config=GEMINI_CONFIG)
    rows.append({"task": task, "model": MODEL, "answer": (gr.text or "").strip(),
                 "seconds": round(time.perf_counter() - start, 2), "ok": check(gr.text or "")})
print(len(rows), "answers")

4 answers


Replay returns Gemini's answer instantly, so its measured time means nothing here (lesson 11). The
real Gemini timings were saved to `../data/recorded/16.json` when the calls were recorded; this cell
loads that recorded output.

In [10]:
import json
from pathlib import Path

RECORDED = Path("../data/recorded/16.json")
saved = json.loads(RECORDED.read_text()) if RECORDED.exists() else {}
if llm_replay.mode() in ("record", "live"):
    saved["rows"] = rows
    RECORDED.write_text(json.dumps(saved, indent=1))
else:
    for row, old in zip(rows, saved["rows"]):
        row["seconds"] = old["seconds"]
for row in rows:
    print(f"{row['task']:<7}{row['model']:<23}{'pass' if row['ok'] else 'FAIL':<5}{row['seconds']:>6.2f}s  {row['answer'][:70]}")

lookup granite4.1:3b          pass   0.35s  5
lookup gemini-3.5-flash-lite  pass   0.61s  Note 5.
math   granite4.1:3b          FAIL   3.07s  24.39 Earth hours.
math   gemini-3.5-flash-lite  FAIL   1.07s  The notes do not cover how many Earth hours 10 Martian days take.


The look-up is easy for both. Both small models fail the arithmetic, but differently. Gemini's
flash-lite declines, which is safe. Granite gives a confident wrong number: its answer looks like it read 39
minutes as 0.39 hours and never multiplied by 10. A wrong number that looks right is the worse failure.
Lesson 17 shows bigger hosted models solving it. Speed is mixed: granite was faster on the one-token
look-up (0.35 s vs 0.61 s) but slower on the math task (3.07 s vs 1.07 s), and slower on average
(1.71 s vs 0.84 s) on this CPU.

### 9. The OpenAI-compatible endpoint
Ollama also serves the OpenAI request format under `/v1`. So the `openai` SDK works if you change two
things: `base_url`, and a dummy `api_key` (the SDK needs one; Ollama ignores it).

In [11]:
from openai import OpenAI

LOCAL_KEY = "local"      # the SDK insists on a key; local servers ignore it

def ask_compatible(base_url: str, model: str, question: str) -> str:
    oa = OpenAI(base_url=base_url, api_key=LOCAL_KEY)
    resp = oa.chat.completions.create(model=model, temperature=0, max_tokens=200, messages=[
        {"role": "system", "content": SYSTEM_TEXT}, {"role": "user", "content": question}])
    u = resp.usage
    return f"{resp.choices[0].message.content!r} | {resp.choices[0].finish_reason} | {u.prompt_tokens} in, {u.completion_tokens} out"

print(ask_compatible("http://localhost:11434/v1", LOCAL_MODEL, TASKS["lookup"][0]))

'5' | stop | 161 in, 2 out


Same answer, now in OpenAI's shape: `choices[0].message.content`, `finish_reason`, `usage`. Code written
for this shape can switch between Ollama, LM Studio and many hosted providers by changing one URL.

### 10. LM Studio: the same function, another port
LM Studio is a desktop app: you pick a GGUF model in its window and start its server from the
Developer tab. It listens on `http://localhost:1234/v1`. Our function needs no change, only the URL
and LM Studio's model id. Skipped unless you enable `download` and have LM Studio running.

In [12]:
try:
    lm_models = [m.id for m in OpenAI(base_url="http://localhost:1234/v1", api_key=LOCAL_KEY).models.list()]
    print("LM Studio models:", lm_models)
    print(ask_compatible("http://localhost:1234/v1", lm_models[0], TASKS["lookup"][0]))
except Exception as e:                       # not installed or not running: the rest of the lesson is unaffected
    print("LM Studio not reachable:", type(e).__name__)

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


### 11. Does the small model pick the right tool?
A **tool** (lesson 01) is a function your code runs when the model asks for it. Lesson 22 builds the full
round trip; here we only check *which* call the model asks for. Ollama turns plain Python functions with
docstrings into tool descriptions. We define two and never run them.

In [13]:
def search_notes(query: str) -> list:
    """Search Scout's notes for one keyword.

    Args:
      query: a single word to look for, such as a planet name
    """

def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression.

    Args:
      expression: arithmetic using numbers and + - * / only, such as '24.65 * 10'
    """

TOOL_CASES = [   # (question, the tool a good assistant would ask for, or None for "just answer")
    ("What do my notes say about Saturn?", "search_notes"),
    ("What is 17.5 times 12?", "calculator"),
    ("Find my notes about the rover.", "search_notes"),
    ("Hi Scout, how are you?", None),
    ("What is the capital of France?", None),
]
print(len(TOOL_CASES), "cases")

5 cases


Each case goes to the model with both tools. `message.tool_calls` holds the requested calls, or is
`None` when the model answers in text.

In [14]:
TOOL_SYSTEM = "You are Scout, a research assistant over the user's notes. Use a tool only when it is needed."
tool_results = []
for question, expected in TOOL_CASES:
    resp = local.chat(model=LOCAL_MODEL, tools=[search_notes, calculator], options={"temperature": 0, "num_predict": 120},
                      messages=[{"role": "system", "content": TOOL_SYSTEM}, {"role": "user", "content": question}])
    calls = [(c.function.name, dict(c.function.arguments)) for c in resp.message.tool_calls or []]
    chosen = calls[0][0] if calls else None
    tool_results.append({"question": question, "expected": expected, "chosen": chosen, "calls": calls})
    print(f"{'ok  ' if chosen == expected else 'MISS'} {question:<38} -> {calls or 'answered in text'}")

ok   What do my notes say about Saturn?     -> [('search_notes', {'query': 'Saturn'})]
ok   What is 17.5 times 12?                 -> [('calculator', {'expression': '17.5 * 12'})]
ok   Find my notes about the rover.         -> [('search_notes', {'query': 'rover'})]
MISS Hi Scout, how are you?                 -> [('search_notes', {'query': 'hello'})]
MISS What is the capital of France?         -> [('search_notes', {'query': 'France'})]


Look at the misses. The arguments it writes are fine, but the model calls a tool even when none is
needed, searching the notes for a greeting. Each needless call costs a step and puts junk into the
context. We save these results next to the timings for the Try it.

In [15]:
if llm_replay.mode() in ("record", "live"):
    saved["tool_results"] = tool_results
    RECORDED.write_text(json.dumps(saved, indent=1))
needed = [t for t in tool_results if t["expected"]]
not_needed = [t for t in tool_results if not t["expected"]]
print("right tool when one was needed:", sum(t["chosen"] == t["expected"] for t in needed), "/", len(needed))
print("no tool when none was needed  :", sum(t["chosen"] is None for t in not_needed), "/", len(not_needed))

right tool when one was needed: 3 / 3
no tool when none was needed  : 0 / 2


### 12. The verdict
Put it together. Local cost is $0 per token (you pay in hardware and electricity); Gemini's is in
lesson 12.

In [16]:
def verdict(rows: list[dict], tool_results: list[dict]) -> None:
    for model in (LOCAL_MODEL, MODEL):
        mine = [r for r in rows if r["model"] == model]
        avg = sum(r["seconds"] for r in mine) / len(mine)
        print(f"{model:<23} passed {sum(r['ok'] for r in mine)}/{len(mine)} tasks, avg {avg:.2f}s per answer")
    hits = sum(t["chosen"] == t["expected"] for t in tool_results)
    print(f"{LOCAL_MODEL:<23} tool choice {hits}/{len(tool_results)} right")

verdict(rows, tool_results)

granite4.1:3b           passed 1/2 tasks, avg 1.71s per answer
gemini-3.5-flash-lite   passed 1/2 tasks, avg 0.84s per answer
granite4.1:3b           tool choice 3/5 right


A fair summary: the local model is good enough for easy look-ups, private notes and offline
development. It is slower on average on this CPU (though not on every question), answers multi-step
arithmetic wrongly instead of declining, and is over-eager with tools.
Bigger local models (`granite4.1:8b`, 5.3 GB) close some of the gap at the cost of more memory.

## At work
- **One client, many backends.** Teams point the OpenAI-compatible client at Ollama, LM Studio or a
  hosted provider by changing `base_url`, so the same code runs offline in development and hosted in
  production. Tool and structured-output fidelity still varies by model. → lesson 37
- **Small model first, escalate when needed.** Routing easy requests to a small or local model and hard
  ones to a large model is a standard cost lever. → lessons 17 and 54
- **Measure before you swap.** A model that passes a demo can fail the multi-step case, as granite did on
  arithmetic; teams run the same eval set on every candidate model. → lesson 48
- **Local is a fallback, not a guarantee.** A local model can keep a degraded service running when a
  provider is down, but its answers are weaker, so the fallback must be tested too. → lesson 57

## What just happened
- Ollama served the open-weight `granite4.1:3b` on `localhost:11434`, in GGUF format, quantised to `Q4_K_M`.
- By hand, 4-bit integers plus one scale stored 8 weights with a small error; that is why 3.4B weights fit in about 2 GB.
- `local.chat` took lesson 08's messages; its reply carried text, a stop reason, token counts and timings in nanoseconds.
- The cold first call spent most of its time loading weights.
- On the same questions, both models answered the look-up; on the arithmetic Gemini flash-lite declined and granite gave a confident wrong number.
- The `openai` SDK talked to Ollama's `/v1` by changing `base_url`; LM Studio works the same way on port 1234.
- The small model chose the right tool when one was needed but also called tools when none was needed.

## Common mistakes
**1. Reading an Ollama reply with OpenAI field names.** The `ollama` client returns its own shape, not
OpenAI's `choices`.

In [17]:
print(r.choices[0].message.content)

AttributeError: 'ChatResponse' object has no attribute 'choices'

Fix: with the `ollama` client read `r.message.content`; use `choices[0].message.content` only on replies
from the `openai` client (step 9).

In [18]:
print(r.message.content)

5


**2. Assuming `tool_calls` is always a list.** When the model answers in text, it is `None`, and looping
over it crashes. Step 6's reply had no tool calls:

In [19]:
for call in r.message.tool_calls:
    print(call.function.name)

TypeError: 'NoneType' object is not iterable

Fix: write `r.message.tool_calls or []`, as step 11 did.

In [20]:
print([c.function.name for c in r.message.tool_calls or []])

[]


**3. Treating Ollama's durations as seconds.** They are nanoseconds, so `eval_count / eval_duration`
gives a tiny number. A sanity check catches it:

In [21]:
tok_per_s = r.eval_count / r.eval_duration
assert tok_per_s > 0.1, f"{tok_per_s} tokens/s is not plausible: durations are in nanoseconds"

AssertionError: 1.2123563357742108e-08 tokens/s is not plausible: durations are in nanoseconds

Fix: divide durations by `1e9` first, as `timings` does.

In [22]:
print(round(r.eval_count / (r.eval_duration / NS), 1), "tokens/s")

12.1 tokens/s


## Try it
Using only `tool_results` (no new model calls), write `tool_report(results)` that returns a dict with
three counts: `wrong_tool` (a tool was needed but another or none was chosen), `needless_call` (no tool
was needed but one was called) and `correct`. Print it. Then add a sixth case of your own to
`TOOL_CASES` and say which bucket you expect it to fall into.

In [23]:
# your code here

Finally, stop the recorder.

In [24]:
llm_replay.stop()

## Key terms
- **open-weight model** — a model whose weights you can download and run yourself.
- **local inference** — running a model on your own machine: no key or per-token bill, data stays local.
- **Ollama** — a local model runner: CLI plus a server on `localhost:11434`, with its own API and an OpenAI-compatible `/v1`.
- **LM Studio** — a desktop app that runs GGUF models and serves an OpenAI-compatible API on `localhost:1234/v1`.
- **quantisation** — storing weights as small integers plus scales (about 4 bits instead of 16) to save memory, at a small cost in accuracy.
- **GGUF** — the single-file model format from llama.cpp that Ollama and LM Studio load.
- **OpenAI-compatible endpoint** — a server that accepts OpenAI's request format, so the `openai` SDK works by changing `base_url`.
- **cold start** — the time a local runner spends loading weights before the first reply.